In [1]:
import pandas as pd
import networkx as nx
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np
import bokeh

Bla  bla bla ask Daniel how he filtered and what he did with data prep

In [2]:
df = pd.read_csv('bgg_data_named_na_years_removed.csv')

What interactivity:
1. Search bar - highlights so we don't need to search for ours if we know
2. zoom
3. pan
4. select - colours the neigbhours\edges and darkens everything else
         AND it filters - i.e. cat filters -> mechanics -> all others
         AND we can            cat filters <- mechanics <- all others
         those not in or not part just get dimmed  
5. brush and link? Link for sure brush prolly

# Mechanics

Because I think with 100k (and more mechancis might appear idk) nodes just a graph is not useful and maybe having a matrix visualisation would essentially give us how to make the graph if need be - and then to explore the visuallisation we would see from the matrix grouping where the "clusters" are and then could either select points from that for extra view or in a search bar enter a thing that we want to highlight. 

In [3]:
df["mechanic_splt"] = df["mechanic"].str.split(",").apply(
    lambda x: [y.strip() for y in x]
)

df["mechanic_splt"] = df["mechanic_splt"].apply(sorted)

edge_counts = {}
mech_degree = {}

for game_mechanics in df["mechanic_splt"]:
    for a, b in combinations(sorted(set(game_mechanics)), 2):
        mech_degree[a] = mech_degree.get(a, 0) + 1
        mech_degree[b] = mech_degree.get(b, 0) + 1
        edge = (a,b)
        edge_counts[edge] = edge_counts.get(edge, 0) + 1

In [4]:
G = nx.Graph()

G.add_weighted_edges_from(
    (node1, node2, weight)
    for (node1, node2), weight in edge_counts.items()
)

communities = nx.community.louvain_communities(
    G,
    weight="weight",
    resolution=1,
    seed=42
)


In [5]:
degrees_of_G = []
for i in G.degree():
    degrees_of_G.append(i[1])

max_deg_G = max(degrees_of_G)

In [6]:
edge_counts_list = list(edge_counts.values())
max_edge_w_G = max(edge_counts_list)

# V2

In [7]:
node_to_community = {}

for i, community in enumerate(communities):
    for node in community:
        node_to_community[node] = i

CG = nx.Graph()

CG.add_nodes_from(range(len(communities)))

for u, v, data in G.edges(data=True):
    cu = node_to_community[u]
    cv = node_to_community[v]

    if cu != cv:
        w = data.get("weight", 1)

        if CG.has_edge(cu, cv):
            CG[cu][cv]["weight"] += w
        else:
            CG.add_edge(cu, cv, weight=w)

In [8]:
community_positions = nx.spring_layout(
    CG,
    weight="weight",
    k=2,
    iterations=100,
    seed=66
)

positions = {}

for i, community in enumerate(communities):

    subgraph = G.subgraph(community)

    local_positions = nx.spring_layout(
        subgraph,
        weight="weight",
        iterations=50,
        seed=66
    )

    center = community_positions[i]

    # Scale based on community size
    radius = np.sqrt(len(community)) * 0.1

    for node, pos in local_positions.items():
        positions[node] = (
            center[0] + radius * pos[0],
            center[1] + radius * pos[1]
        )

# Assign a color to each community
node_colors = [
    node_to_community[node]
    for node in G.nodes()
]

# Plot
plt.figure(figsize=(20, 20))

edge_colors = [
    (0, 0, 0, 0.01 + 0.99 * G[u][v]["weight"] / max_edge_w_G)
    for u, v in G.edges()
]




<Figure size 2000x2000 with 0 Axes>

# Interact

In [9]:

import networkx as nx
import numpy as np

from bokeh.io import show
from bokeh.models import (
    GraphRenderer,
    StaticLayoutProvider,
    ColumnDataSource,
    HoverTool,
    NodesAndLinkedEdges,
    CustomJS,
    Range1d,
    Scatter,
    MultiLine, TapTool, MultiSelect
)
from bokeh.plotting import figure
from bokeh.palettes import Spectral4


# --------------------------------------------------
# 1. Example graph (replace with your own G and pos)
# --------------------------------------------------

#G = nx.karate_club_graph()
#pos = nx.spring_layout(G, seed=42)

pos = positions

# --------------------------------------------------
# 2. Prepare node data
# --------------------------------------------------

nodes = list(G.nodes())

node_source = ColumnDataSource(data=dict(
    index=nodes,
    x=[pos[n][0] for n in nodes],
    y=[pos[n][1] for n in nodes],
    degree=[G.degree(n) for n in nodes],

    # Add any custom attributes you want
    label=[str(n) for n in nodes],
    info=[f"Information about node {n}" for n in nodes],

    # Initial visual properties
    fill_color=["#4C78A8"] * len(nodes),
    line_color=["#333333"] * len(nodes),
    alpha= [0.6] * len(nodes),
    size=[15] * len(nodes),
))

# --------------------------------------------------
# 3. Prepare edge data
# --------------------------------------------------

edges = list(G.edges())

edge_colors = [
    7 * G[u][v]["weight"] / max_edge_w_G 
    for u, v in G.edges()
]

edge_source = ColumnDataSource(data=dict(
    start=[u for u, v in edges],
    end=[v for u, v in edges],
    line_color=["#999999"] * len(edges),
    alpha= edge_colors, # [0.7] * len(edges),
    line_width=[1.5] * len(edges),

    original_alpha = edge_colors,
    original_line_width=[1.5] * len(edges),
))

# --------------------------------------------------
# 4. Create graph renderer
# --------------------------------------------------

graph = GraphRenderer()

graph.node_renderer.data_source = node_source
graph.edge_renderer.data_source = edge_source


graph.layout_provider = StaticLayoutProvider(graph_layout=pos)

graph.node_renderer.glyph = Scatter(
    marker="circle",
    size="size",
    fill_color="fill_color",
    line_color="line_color",
    fill_alpha="alpha",
    line_alpha="alpha",
)

graph.edge_renderer.glyph = MultiLine(
    line_color="line_color",
    line_alpha="alpha",
    line_width="line_width",
)


'''
graph.node_renderer.glyph = {
    "fill_color": "fill_color",
    "line_color": "line_color",
    "fill_alpha": "alpha",
    "line_alpha": "alpha",
    "size": "size",
}

graph.edge_renderer.glyph = {
    "line_color": "line_color",
    "line_alpha": "alpha",
    "line_width": "line_width",
}
'''
# --------------------------------------------------
# 5. Plot
# --------------------------------------------------

p = figure(
    width=900,
    height=700,
    title="Interactive Network",
    tools="pan,wheel_zoom,box_zoom,reset,save",
    active_scroll="wheel_zoom",
    x_range=Range1d(-1.2, 1.2),
    y_range=Range1d(-1.2, 1.2),
)

p.renderers.append(graph)

# FIX UNSLELECTING
#tap = p.select_one(TapTool)
#tap.renderers = [graph.node_renderer]
#tap.mode = "toggle"


# --------------------------------------------------
# 6. Hover information
# --------------------------------------------------

hover = HoverTool(
    renderers=[graph.node_renderer],
    tooltips=[
        ("Node", "@label"),
        ("Degree", "@degree"),
        ("Information", "@info"),
    ],
)

p.add_tools(hover)

# --------------------------------------------------
# 7. Selection behaviour
# --------------------------------------------------


callback = CustomJS(
    args=dict(
        nodes=node_source,
        edges=edge_source,
        xr=p.x_range,
        yr=p.y_range,
    ),
    code="""
    const nd = nodes.data;
    const ed = edges.data;

    // ------------------------------------------------
    // 1. Identify the node closest to the click
    // ------------------------------------------------

    const x = cb_obj.x;
    const y = cb_obj.y;

    const plot_width = 900;
    const plot_height = 700;

    const x_span = xr.end - xr.start;
    const y_span = yr.end - yr.start;

    let clicked = -1;
    let min_dist = Infinity;

    for (let i = 0; i < nd.index.length; i++) {

        const dx = (nd.x[i] - x) * plot_width / x_span;
        const dy = (nd.y[i] - y) * plot_height / y_span;

        const dist = Math.sqrt(dx*dx + dy*dy);

        if (dist < min_dist) {
            min_dist = dist;
            clicked = i;
        }
    }

    // Ignore clicks that are not close to a node
    if (min_dist > 20) return;

    // ------------------------------------------------
    // 2. Toggle clicked node
    // ------------------------------------------------

    let selected = [...nodes.selected.indices];

    if (selected.includes(clicked)) {
        selected = selected.filter(i => i !== clicked);
    }
    else {
        selected.push(clicked);
    }

    nodes.selected.indices = selected;

    // ------------------------------------------------
    // 3. Get selected node IDs
    // ------------------------------------------------

    const selected_ids = new Set(
        selected.map(i => nd.index[i])
    );

    // ------------------------------------------------
    // 4. Find COMMON neighbours (intersection)
    // ------------------------------------------------

    let common = null;

    for (const selected_id of selected_ids) {

        const neighbours = new Set();

        for (let i = 0; i < ed.start.length; i++) {

            const u = ed.start[i];
            const v = ed.end[i];

            if (u === selected_id) neighbours.add(v);
            if (v === selected_id) neighbours.add(u);
        }

        if (common == null) {
            common = neighbours;
        }
        else {
            common = new Set(
                [...common].filter(n => neighbours.has(n))
            );
        }
    }

    if (common === null) common = new Set();

    // Do not highlight selected nodes as neighbours
    for (const id of selected_ids) {
        common.delete(id);
    }


    // ------------------------------------------------
    // 5. Style nodes
    // ------------------------------------------------

    for (let i = 0; i < nd.index.length; i++) {

        const id = nd.index[i];

        if (selected_ids.has(id)) {

            // Selected nodes: RED
            nd.fill_color[i] = "#E45756";
            nd.line_color[i] = "#A52A2A";
            nd.alpha[i] = 1.0;
            nd.size[i] = 22;

        } else if (common.has(id)) {

            // Common neighbours: YELLOW
            nd.fill_color[i] = "#E6AC00";
            nd.line_color[i] = "#B8860B";
            nd.alpha[i] = 1.0;
            nd.size[i] = 18;

        } else if (selected_ids.size > 0) {

            // Everything else: DIMMED
            nd.fill_color[i] = "#BDBDBD";
            nd.line_color[i] = "#BDBDBD";
            nd.alpha[i] = 0.12;
            nd.size[i] = 15;

        }
        else {

            // Nothing selected: RESTORE ORIGINAL
            nd.fill_color[i] = "#4C78A8";
            nd.line_color[i] = "#333333";
            nd.alpha[i] = 0.9;
            nd.size[i] = 15;
        }
    }

    // ------------------------------------------------
    // 6. Style edges
    // ------------------------------------------------

    for (let i = 0; i < ed.start.length; i++) {

        const u = ed.start[i];
        const v = ed.end[i];

        if (
            (selected_ids.has(u) && common.has(v)) ||
            (selected_ids.has(v) && common.has(u))
            ){

            ed.line_color[i] = "#E45756";
            ed.alpha[i] = 0.7+4*ed.original_alpha[i];
            ed.line_width[i] = 3;

        }
        else if (selected_ids.size > 0) {

            ed.line_color[i] = "#BDBDBD";
            ed.alpha[i] = ed.original_alpha[i];
            ed.line_width[i] = 1;

        }
        else {

            ed.line_color[i] = "#999999";
            ed.alpha[i] = ed.original_alpha[i]; 
            ed.line_width[i] = 1.5;
        }
    }

    nodes.change.emit();
    edges.change.emit();
    """
)

p.js_on_event("tap", callback)

# --------------------------------------------------
# 8. Display
# --------------------------------------------------

show(p)


In [19]:
print(bokeh.__version__)

3.10.2
